# L14 — Avellaneda-Stoikov: El Market Maker Óptimo
## Ejercicios

| Tier | Ejercicios | Cuándo |
|------|-----------|--------|
| **Núcleo** | E1–E5 | Completar en clase |
| **Si vamos bien** | E6–E7 | Si el ritmo lo permite |
| **Bonus / casa** | E8–E10 | Tarea o quien termine antes |

> **Continuidad:** Este notebook reutiliza `price_path`, `NaiveMarketMaker`, `SkewedMarketMaker`,
> `reservation_price` y `MarketMakingBacktest` de L13. Cópialos (o importa desde L13) antes de empezar E3–E5.


## Setup — constantes y utilidades de L13

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Constantes compartidas con L13 ──────────────────────────────
SIGMA    = 0.05     # volatilidad del precio por step
SIGMA_SQ = 0.0025   # SIGMA**2
SPREAD   = 0.10     # spread total naive
KAPPA    = 5.0      # decaimiento de fill prob: p = exp(-κ·δ)
ARR      = 1.0      # tasa de llegada base (órdenes/step)
DT       = 1.0
T_TOTAL  = 3600     # duración de la sesión (steps)
MID0     = 100.0    # precio inicial

# ── price_path (copiado de L13) ──────────────────────────────────
def price_path(T=100, dt=1, sigma=SIGMA, seed=42):
    """Genera un camino de precios con movimiento browniano aritmético."""
    rng = np.random.default_rng(seed)
    path = [MID0]
    for _ in range(T):
        path.append(path[-1] + sigma * rng.standard_normal() * np.sqrt(dt))
    return path

# ── NaiveMarketMaker (copiado de L13) ───────────────────────────
class NaiveMarketMaker:
    def __init__(self, spread=SPREAD, arr=ARR, seed=42):
        self.spread = spread
        self.arr = arr
        self._rng = np.random.default_rng(seed)
        self.cash = 0.0
        self.inventory = 0
        self.fills = 0
        self.inv_hist = []

    def run(self, price_path_data):
        for mid in price_path_data[:-1]:
            half = self.spread / 2
            bid  = mid - half
            ask  = mid + half
            p    = np.exp(-KAPPA * half) * self.arr * DT
            if self._rng.random() < p:   # ask fill → sell
                self.inventory -= 1; self.cash += ask; self.fills += 1
            if self._rng.random() < p:   # bid fill → buy
                self.inventory += 1; self.cash -= bid; self.fills += 1
            self.inv_hist.append(self.inventory)
        return self

    def pnl(self, last_mid):
        return self.cash + self.inventory * last_mid

# ── reservation_price (copiado de L13 — versión estática) ───────
def reservation_price(mid, q, gamma, sigma_sq=SIGMA_SQ):
    """Precio de reserva estático (sin factor T−t). Usado en SkewedMarketMaker."""
    return mid - q * gamma * sigma_sq

# ── SkewedMarketMaker (copiado de L13) ──────────────────────────
class SkewedMarketMaker:
    def __init__(self, spread=SPREAD, gamma=0.1, arr=ARR, seed=42):
        self.spread = spread
        self.gamma = gamma
        self.arr = arr
        self._rng = np.random.default_rng(seed)
        self.cash = 0.0
        self.inventory = 0
        self.fills = 0
        self.inv_hist = []

    def run(self, price_path_data):
        for mid in price_path_data[:-1]:
            half = self.spread / 2
            res  = reservation_price(mid, self.inventory, self.gamma, SIGMA_SQ)
            bid  = res - half
            ask  = res + half
            d_bid = max(mid - bid, 0)
            d_ask = max(ask - mid, 0)
            p_bid = np.exp(-KAPPA * d_bid) * self.arr * DT
            p_ask = np.exp(-KAPPA * d_ask) * self.arr * DT
            if self._rng.random() < p_ask:
                self.inventory -= 1; self.cash += ask; self.fills += 1
            if self._rng.random() < p_bid:
                self.inventory += 1; self.cash -= bid; self.fills += 1
            self.inv_hist.append(self.inventory)
        return self

    def pnl(self, last_mid):
        return self.cash + self.inventory * last_mid

# ── MarketMakingBacktest (copiado de L13 E10) ───────────────────
class MarketMakingBacktest:
    def __init__(self, T=T_TOTAL, N=20):
        self.T = T
        self.N = N

    def run(self, MMClass, **kwargs):
        results = []
        for i in range(self.N):
            path = price_path(T=self.T, seed=i)
            mm = MMClass(seed=i, **kwargs)
            mm.run(path)
            inv_arr = np.array(mm.inv_hist)
            results.append({
                'run': i,
                'fills': mm.fills,
                'final_inv': mm.inventory,
                'pnl': mm.pnl(path[-1]),
                'inv_std': np.std(inv_arr),
                'max_abs_inv': int(np.max(np.abs(inv_arr)))
            })
        return pd.DataFrame(results)

    def compare(self, mm_classes, labels, **shared_kwargs):
        dfs = []
        for cls, lbl in zip(mm_classes, labels):
            df = self.run(cls, **shared_kwargs)
            df['model'] = lbl
            dfs.append(df)
        return pd.concat(dfs, ignore_index=True)

print("Setup completo — constantes y clases de L13 disponibles.")


---
## Ejercicio 0 — Motivación (sin código)

**Reflexiona antes de empezar:**

1. En L13 el `SkewedMarketMaker` reduce el inventario usando `reservation_price(mid, q, γ, σ²)`.
   ¿Qué le falta a esa fórmula respecto a la versión completa de Avellaneda-Stoikov?

2. El `SkewedMarketMaker` usa siempre `SPREAD/2` como half-spread. ¿Por qué eso no es óptimo?
   ¿Qué debería depender el spread óptimo?

3. A medida que se acerca el cierre de sesión (T−t → 0), ¿qué debería hacer un MM con inventario?
   ¿Ampliar o estrechar el spread? ¿Subir o bajar el precio de reserva?

> Después de los ejercicios, vuelve y responde estas preguntas.


---
## Ejercicio 1 — Precio de reserva de Avellaneda-Stoikov ⭐

Implementa la versión completa del precio de reserva que incluye el factor temporal T−t:

```
r(s, q, γ, σ², τ) = s − q · γ · σ² · τ
```

donde τ = T − t es el tiempo restante hasta el cierre de sesión.

La diferencia con L13: ahí τ no existía (era efectivamente 1). Aquí el sesgo
crece con el horizonte temporal — al inicio de sesión, el MM se aleja más del mid;
al final, las cotizaciones convergen al mid.


In [ ]:
def as_reservation_price(mid, q, gamma, sigma_sq, T_minus_t):
    """
    Precio de reserva de Avellaneda-Stoikov.

    Parámetros:
        mid       : mid-price actual
        q         : inventario actual (+ largo, − corto)
        gamma     : aversión al riesgo del market maker
        sigma_sq  : varianza del precio por step (SIGMA**2)
        T_minus_t : tiempo restante hasta cierre (τ = T − t)

    Retorna:
        r : precio de reserva
    """
    pass


In [ ]:
# ── Validación E1 ───────────────────────────────────────────────
assert 'as_reservation_price' in globals(), "Falta definir as_reservation_price"

cases = [
    (100.0, 5,   0.1, SIGMA_SQ, T_TOTAL, 95.5),
    (100.0, -3,  0.1, SIGMA_SQ, T_TOTAL, 102.7),
    (100.0, 0,   0.1, SIGMA_SQ, 0,       100.0),
    (100.0, 10,  0.2, SIGMA_SQ, T_TOTAL, 82.0),
]
for mid, q, g, sq, tau, expected in cases:
    got = as_reservation_price(mid, q, g, sq, tau)
    assert abs(got - expected) < 1e-9, \
        f"as_reservation_price({mid},{q},{g},{sq},{tau}) = {got:.6f}, esperado {expected}"

# Propiedad: al final de sesión (τ=0), r = mid independientemente de q
r_final = as_reservation_price(100.0, 50, 0.9, SIGMA_SQ, 0)
assert abs(r_final - 100.0) < 1e-9, f"A T−t=0 debe ser r=mid. Obtenido: {r_final}"

print("E1 ✓ — as_reservation_price correcta")
print(f"  Ejemplo: q=5, γ=0.1, τ=3600 → r = {as_reservation_price(100,5,0.1,SIGMA_SQ,3600):.4f}")
print(f"  (sesgo = {as_reservation_price(100,5,0.1,SIGMA_SQ,3600)-100:.4f} respecto al mid)")


**Solución E1:**
```python
def as_reservation_price(mid, q, gamma, sigma_sq, T_minus_t):
    return mid - q * gamma * sigma_sq * T_minus_t
```


---
## Ejercicio 2 — Half-spread óptimo de Avellaneda-Stoikov ⭐

El half-spread óptimo tiene dos componentes independientes:

```
δ*(γ, σ², κ, τ) = γ·σ²·τ/2  +  (1/γ)·ln(1 + γ/κ)
                  ─────────────   ───────────────────
                  cobertura inv.   liquidez permanente
```

- **Cobertura de inventario** `γ·σ²·τ/2`: decrece linealmente con τ.
  A inicio de sesión (τ grande) el spread es amplio — el MM cubre el riesgo de inventario.
  A cierre de sesión (τ=0) este término desaparece.

- **Liquidez permanente** `(1/γ)·ln(1 + γ/κ)`: constante, no depende de τ.
  Es la compensación mínima que el MM exige por proveer liquidez,
  incluso en el último instante de la sesión.


In [ ]:
def as_optimal_halfspread(gamma, sigma_sq, kappa, T_minus_t):
    """
    Half-spread óptimo de Avellaneda-Stoikov.

    Retorna un dict con:
        'inv'   : componente de cobertura de inventario
        'liq'   : componente de liquidez permanente
        'total' : suma de ambos = δ*
    """
    pass


In [ ]:
# ── Validación E2 ───────────────────────────────────────────────
assert 'as_optimal_halfspread' in globals(), "Falta definir as_optimal_halfspread"

r1 = as_optimal_halfspread(0.1, SIGMA_SQ, KAPPA, T_TOTAL)
assert isinstance(r1, dict) and 'inv' in r1 and 'liq' in r1 and 'total' in r1, \
    "Debe retornar dict con claves 'inv', 'liq', 'total'"

assert abs(r1['inv']   - 0.45000) < 1e-5, f"inv={r1['inv']}, esperado 0.45000"
assert abs(r1['liq']   - 0.19803) < 1e-5, f"liq={r1['liq']}, esperado 0.19803"
assert abs(r1['total'] - 0.64803) < 1e-5, f"total={r1['total']}, esperado 0.64803"

# A τ=0: solo queda la componente permanente
r_T = as_optimal_halfspread(0.1, SIGMA_SQ, KAPPA, 0)
assert abs(r_T['inv'])              < 1e-9, "A τ=0 la componente inv debe ser 0"
assert abs(r_T['liq']   - 0.19803) < 1e-5, "La componente liq no depende de τ"
assert abs(r_T['total'] - 0.19803) < 1e-5, f"total a τ=0 = {r_T['total']}"

# Propiedad: la componente liq es igual a τ=3600 y τ=0
assert abs(r1['liq'] - r_T['liq']) < 1e-9, "La componente liq debe ser igual para cualquier τ"

print("E2 ✓ — as_optimal_halfspread correcta")
print(f"  τ=3600: inv={r1['inv']:.5f}, liq={r1['liq']:.5f}, total={r1['total']:.5f}")
print(f"  τ=0:    inv={r_T['inv']:.5f}, liq={r_T['liq']:.5f}, total={r_T['total']:.5f}")
print(f"  La componente liq es el spread mínimo: {r_T['liq']:.5f}")


**Solución E2:**
```python
def as_optimal_halfspread(gamma, sigma_sq, kappa, T_minus_t):
    inv = gamma * sigma_sq * T_minus_t / 2
    liq = (1 / gamma) * np.log(1 + gamma / kappa)
    return {'inv': inv, 'liq': liq, 'total': inv + liq}
```


---
## Ejercicio 3 — Clase ASMarketMaker ⭐

Implementa el market maker de Avellaneda-Stoikov. La diferencia clave con `SkewedMarketMaker`:

1. Usa `as_reservation_price` con el factor τ = T − t dinámico
2. Usa `as_optimal_halfspread` para calcular δ* en cada step (no un spread fijo)
3. La probabilidad de fill es asimétrica: `p_ask = exp(−κ · max(ask − mid, 0))` y análoga para bid

El parámetro `T` es el número de steps totales de la sesión. En el step t, τ = T − t.

**Orden de fills**: ask primero (reduce inventario), luego bid (aumenta inventario).


In [ ]:
class ASMarketMaker:
    def __init__(self, T=T_TOTAL, gamma=0.1, arr=ARR, seed=42):
        self.T       = T
        self.gamma   = gamma
        self.arr     = arr
        self._rng    = np.random.default_rng(seed)
        self.cash      = 0.0
        self.inventory = 0
        self.fills     = 0
        self.inv_hist  = []

    def run(self, price_path_data):
        """
        Ejecuta el market maker sobre un camino de precios.

        En cada step t (0-indexed):
          - tau  = self.T - t
          - r    = as_reservation_price(mid, q, gamma, SIGMA_SQ, tau)
          - d    = as_optimal_halfspread(gamma, SIGMA_SQ, KAPPA, tau)['total']
          - bid  = r - d
          - ask  = r + d
          - p_ask = exp(-KAPPA * max(ask - mid, 0)) * arr * DT
          - p_bid = exp(-KAPPA * max(mid - bid, 0)) * arr * DT
          - ask fill primero (inventory -= 1, cash += ask)
          - bid fill segundo (inventory += 1, cash -= bid)

        Guarda el inventario en self.inv_hist tras cada step.
        """
        pass

    def pnl(self, last_mid):
        """P&L mark-to-market al precio last_mid."""
        return self.cash + self.inventory * last_mid


In [ ]:
# ── Validación E3 (carrera rápida) ─────────────────────────────
assert 'ASMarketMaker' in globals(), "Falta definir ASMarketMaker"
path_short = price_path(T=100, seed=42)
mm_test = ASMarketMaker(T=100, gamma=0.1, seed=42)
mm_test.run(path_short)
assert hasattr(mm_test, 'inv_hist'), "run() debe poblar self.inv_hist"
assert len(mm_test.inv_hist) == 100, f"inv_hist debe tener 100 entradas, tiene {len(mm_test.inv_hist)}"
assert isinstance(mm_test.fills, int) and mm_test.fills >= 0, "fills debe ser entero >= 0"
print(f"E3 ✓ — ASMarketMaker instancia correctamente (T=100, fills={mm_test.fills})")


**Solución E3:**
```python
def run(self, price_path_data):
    for t, mid in enumerate(price_path_data[:-1]):
        tau   = self.T - t
        r     = as_reservation_price(mid, self.inventory, self.gamma, SIGMA_SQ, tau)
        delta = as_optimal_halfspread(self.gamma, SIGMA_SQ, KAPPA, tau)['total']
        bid   = r - delta
        ask   = r + delta
        d_ask = max(ask - mid, 0)
        d_bid = max(mid - bid, 0)
        p_ask = np.exp(-KAPPA * d_ask) * self.arr * DT
        p_bid = np.exp(-KAPPA * d_bid) * self.arr * DT
        if self._rng.random() < p_ask:
            self.inventory -= 1; self.cash += ask; self.fills += 1
        if self._rng.random() < p_bid:
            self.inventory += 1; self.cash -= bid; self.fills += 1
        self.inv_hist.append(self.inventory)
```


---
## Ejercicio 4 — Validación completa del ASMarketMaker ⭐

Corre el `ASMarketMaker` sobre `price_path(T=3600, seed=42)` con γ=0.1.
Verifica que los resultados clave coinciden con los valores esperados.


In [ ]:
# Genera el camino de 3600 steps
path_3600 = price_path(T=3600, seed=42)

mm_as = ASMarketMaker(T=T_TOTAL, gamma=0.1, seed=42)
mm_as.run(path_3600)

print(f"fills     = {mm_as.fills}")
print(f"inventory = {mm_as.inventory}")
print(f"P&L       = {mm_as.pnl(path_3600[-1]):.4f}")
print(f"inv_std   = {np.std(mm_as.inv_hist):.4f}")
print(f"max_abs   = {max(abs(i) for i in mm_as.inv_hist)}")


In [ ]:
# ── Validación E4 ───────────────────────────────────────────────
assert mm_as.fills == 1414, \
    f"fills={mm_as.fills}, esperado 1414"
assert mm_as.inventory == 0, \
    f"inventory={mm_as.inventory}, esperado 0"
assert abs(mm_as.pnl(path_3600[-1]) - 301.3136) < 1.0, \
    f"P&L={mm_as.pnl(path_3600[-1]):.4f}, esperado ≈301.31 (tolerancia 1.0)"
inv_arr = np.array(mm_as.inv_hist)
assert abs(np.std(inv_arr) - 0.5918) < 0.05, \
    f"inv_std={np.std(inv_arr):.4f}, esperado ≈0.5918"
assert max(abs(i) for i in mm_as.inv_hist) == 4, \
    f"max_abs={max(abs(i) for i in mm_as.inv_hist)}, esperado 4"

print("E4 ✓ — ASMarketMaker validado")
print(f"  El inventario nunca superó ±4 lotes durante 3600 steps.")


**Solución E4:** No hay código adicional — la solución es la implementación de E3 correcta.


---
## Ejercicio 5 — Comparación triple: Naive / Skewed / A-S ⭐

Usa `MarketMakingBacktest` (20 runs, T=3600) para comparar los tres market makers.
Calcula para cada modelo: fills medio, inv_std medio y max_abs_inv medio.

Criterio de éxito: `inv_std_AS < inv_std_Skewed < inv_std_Naive`.


In [ ]:
bt = MarketMakingBacktest(T=T_TOTAL, N=20)

# Corre los tres modelos
df_comparison = bt.compare(
    mm_classes=[NaiveMarketMaker, SkewedMarketMaker, ASMarketMaker],
    labels=['Naive', 'Skewed', 'A-S'],
    gamma=0.1
)

# Resumen por modelo
summary = df_comparison.groupby('model')[['fills','pnl','inv_std','max_abs_inv']].mean().round(2)
print(summary)


In [ ]:
# ── Validación E5 ───────────────────────────────────────────────
means = df_comparison.groupby('model')['inv_std'].mean()
naive_std  = means.get('Naive',  means.get('naive'))
skewed_std = means.get('Skewed', means.get('skewed'))
as_std     = means.get('A-S',    means.get('as'))

assert naive_std  is not None, "Falta el modelo Naive en df_comparison"
assert skewed_std is not None, "Falta el modelo Skewed en df_comparison"
assert as_std     is not None, "Falta el modelo A-S en df_comparison"
assert as_std < skewed_std < naive_std, \
    f"Orden esperado: AS < Skewed < Naive. Obtenido: {as_std:.3f} / {skewed_std:.3f} / {naive_std:.3f}"

# Verificar que A-S reduce inventario al menos un 90% vs naive
reduction = (1 - as_std / naive_std) * 100
assert reduction > 90, f"Reducción de inventario A-S vs Naive: {reduction:.1f}%, esperado >90%"

print(f"E5 ✓ — AS ({as_std:.3f}) < Skewed ({skewed_std:.3f}) < Naive ({naive_std:.3f})")
print(f"  A-S reduce el inventario std un {reduction:.1f}% respecto al Naive.")


**Solución E5:**
```python
bt = MarketMakingBacktest(T=T_TOTAL, N=20)
df_comparison = bt.compare(
    mm_classes=[NaiveMarketMaker, SkewedMarketMaker, ASMarketMaker],
    labels=['Naive', 'Skewed', 'A-S'],
    gamma=0.1
)
summary = df_comparison.groupby('model')[['fills','pnl','inv_std','max_abs_inv']].mean().round(2)
```


---
## Ejercicio 6 — Sensibilidad a γ (Si vamos bien)

Corre `ASMarketMaker` con γ ∈ {0.01, 0.05, 0.10, 0.20, 0.30, 0.50} sobre `path_3600`.
Para cada γ, calcula: fills, inv_std, max_abs_inv y P&L.

Construye un DataFrame con estos resultados y grafícalo.


In [ ]:
gammas = [0.01, 0.05, 0.10, 0.20, 0.30, 0.50]
rows = []

for g in gammas:
    mm_g = ASMarketMaker(T=T_TOTAL, gamma=g, seed=42)
    mm_g.run(path_3600)
    inv_arr = np.array(mm_g.inv_hist)
    rows.append({
        'gamma': g,
        'fills': mm_g.fills,
        'inv_std': round(np.std(inv_arr), 4),
        'max_abs_inv': int(np.max(np.abs(inv_arr))),
        'pnl': round(mm_g.pnl(path_3600[-1]), 2)
    })

df_gamma = pd.DataFrame(rows)
print(df_gamma.to_string(index=False))


In [ ]:
# ── Validación E6 ───────────────────────────────────────────────
assert 'df_gamma' in globals() and len(df_gamma) == 6, \
    "df_gamma debe tener 6 filas (una por γ)"
assert list(df_gamma['gamma']) == gammas, "Orden de gammas incorrecto"

# Propiedad 1: fills decrece con γ
fills_list = list(df_gamma['fills'])
assert all(fills_list[i] >= fills_list[i+1] for i in range(len(fills_list)-1)), \
    "fills debe ser monótonamente decreciente con γ"

# Propiedad 2: inv_std decrece con γ
std_list = list(df_gamma['inv_std'])
assert all(std_list[i] >= std_list[i+1] for i in range(len(std_list)-1)), \
    "inv_std debe ser monótonamente decreciente con γ"

# Valores spot-check
assert df_gamma.loc[df_gamma.gamma==0.01, 'fills'].values[0] == 2475, "fills(γ=0.01) debe ser 2475"
assert df_gamma.loc[df_gamma.gamma==0.50, 'fills'].values[0] == 341,  "fills(γ=0.50) debe ser 341"

print("E6 ✓ — Sensibilidad γ correcta")
print("  Tradeoff claro: ↑γ → ↓inventario pero también ↓fills y ↓P&L.")


In [ ]:
# Visualización opcional
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.patch.set_facecolor('#09090b')

for ax in axes:
    ax.set_facecolor('#18181b')
    ax.tick_params(colors='#a1a1aa')
    for spine in ax.spines.values():
        spine.set_color('#3f3f46')

axes[0].plot(df_gamma.gamma, df_gamma.fills, marker='o', color='#22d3ee')
axes[0].set_title('Fills vs γ', color='#f4f4f5'); axes[0].set_xlabel('γ', color='#a1a1aa')

axes[1].plot(df_gamma.gamma, df_gamma.inv_std, marker='o', color='#f87171')
axes[1].set_title('Inv std vs γ', color='#f4f4f5'); axes[1].set_xlabel('γ', color='#a1a1aa')

axes[2].plot(df_gamma.gamma, df_gamma.pnl, marker='o', color='#4ade80')
axes[2].set_title('P&L vs γ', color='#f4f4f5'); axes[2].set_xlabel('γ', color='#a1a1aa')

plt.tight_layout()
plt.show()


**Solución E6:** El código en las celdas es la solución completa.


---
## Ejercicio 7 — Shock de volatilidad (Si vamos bien)

Genera un camino de precios con un shock de volatilidad:

```python
def price_path_with_shock(T=3600, sigma_base=SIGMA, sigma_shock=SIGMA*3,
                          shock_start=1200, shock_end=1500, seed=42):
    ...
```

Compara el comportamiento del Naive y el A-S durante y después del shock.
¿Qué pasa con el inventario de cada uno?


In [ ]:
def price_path_with_shock(T=3600, sigma_base=SIGMA, sigma_shock=SIGMA*3,
                          shock_start=1200, shock_end=1500, seed=42):
    """Camino de precios con un bloque de alta volatilidad entre shock_start y shock_end."""
    rng = np.random.default_rng(seed)
    path = [MID0]
    for t in range(T):
        sigma = sigma_shock if shock_start <= t < shock_end else sigma_base
        path.append(path[-1] + sigma * rng.standard_normal())
    return path

path_shock = price_path_with_shock()

# Ejecutar ambos modelos
mm_naive_sh = NaiveMarketMaker(seed=42)
mm_naive_sh.run(path_shock)

mm_as_sh = ASMarketMaker(T=T_TOTAL, gamma=0.1, seed=42)
mm_as_sh.run(path_shock)

# Comparación durante el shock
inv_naive_shock = mm_naive_sh.inv_hist[1200:1500]
inv_as_shock    = mm_as_sh.inv_hist[1200:1500]

print(f"Inventario durante el shock (steps 1200-1500):")
print(f"  Naive std: {np.std(inv_naive_shock):.3f}, max_abs: {max(abs(i) for i in inv_naive_shock)}")
print(f"  A-S   std: {np.std(inv_as_shock):.3f},   max_abs: {max(abs(i) for i in inv_as_shock)}")


In [ ]:
# ── Validación E7 ───────────────────────────────────────────────
assert 'path_shock' in globals(), "Falta definir path_shock"
assert len(path_shock) == T_TOTAL + 1, "path_shock debe tener T+1 puntos"

inv_naive_shock = mm_naive_sh.inv_hist[1200:1500]
inv_as_shock    = mm_as_sh.inv_hist[1200:1500]

# Durante el shock, A-S debe tener menor std que Naive
assert np.std(inv_as_shock) < np.std(inv_naive_shock), \
    "Durante el shock, A-S debe acumular menos inventario que Naive"

print("E7 ✓ — A-S controla mejor el inventario durante el shock de volatilidad")


**Solución E7:** El código en las celdas es la solución completa.


---
## Ejercicio 8 — Evolución de δ*(T−t) a lo largo de la sesión (Bonus)

Genera una tabla con δ* para τ = T, 0.75T, 0.5T, 0.25T, 0 con γ = 0.1.
Muestra la descomposición en componentes inv y liq en cada punto.
Grafica cómo evoluciona el spread a lo largo de la sesión.


In [ ]:
taus = [T_TOTAL, int(0.75*T_TOTAL), int(0.5*T_TOTAL), int(0.25*T_TOTAL), 0]
gamma = 0.1

rows_delta = []
for tau in taus:
    d = as_optimal_halfspread(gamma, SIGMA_SQ, KAPPA, tau)
    rows_delta.append({
        'tau': tau,
        't_pct': f'{100*(T_TOTAL-tau)/T_TOTAL:.0f}% elapsed',
        'delta_inv': round(d['inv'], 5),
        'delta_liq': round(d['liq'], 5),
        'delta_total': round(d['total'], 5)
    })

df_delta = pd.DataFrame(rows_delta)
print(df_delta.to_string(index=False))


In [ ]:
# ── Validación E8 ───────────────────────────────────────────────
assert 'df_delta' in globals() and len(df_delta) == 5
assert abs(df_delta.iloc[0]['delta_total'] - 0.64803) < 1e-4
assert abs(df_delta.iloc[-1]['delta_total'] - 0.19803) < 1e-4
# La componente liq debe ser constante
liqs = df_delta['delta_liq'].values
assert np.allclose(liqs, liqs[0], atol=1e-9), "delta_liq debe ser constante en τ"
print("E8 ✓ — δ*(τ) evoluciona correctamente")


In [ ]:
# Visualización
tau_range = np.linspace(0, T_TOTAL, 500)
inv_comp   = [as_optimal_halfspread(0.1, SIGMA_SQ, KAPPA, t)['inv'] for t in tau_range]
liq_comp   = [as_optimal_halfspread(0.1, SIGMA_SQ, KAPPA, t)['liq'] for t in tau_range]
total_comp = [a+b for a,b in zip(inv_comp, liq_comp)]

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#09090b'); ax.set_facecolor('#18181b')
ax.tick_params(colors='#a1a1aa')
for spine in ax.spines.values(): spine.set_color('#3f3f46')

ax.fill_between(tau_range, liq_comp, total_comp, alpha=0.3, color='#fbbf24', label='Cobertura inventario')
ax.fill_between(tau_range, 0, liq_comp, alpha=0.3, color='#4ade80', label='Liquidez permanente')
ax.plot(tau_range, total_comp, color='#22d3ee', lw=2, label='δ* total')
ax.axhline(liq_comp[0], color='#4ade80', lw=1, ls='--', alpha=0.5)

ax.set_xlabel('Tiempo restante τ = T−t', color='#a1a1aa')
ax.set_ylabel('Half-spread δ*', color='#a1a1aa')
ax.set_title('Evolución del half-spread óptimo', color='#f4f4f5')
ax.legend(facecolor='#18181b', labelcolor='#a1a1aa')
plt.tight_layout()
plt.show()


**Solución E8:** El código en las celdas es la solución completa.

---
## Ejercicio 9 — Calibración de κ a partir de datos de fills (Bonus)

Dado que la probabilidad de fill es `p = exp(−κ·δ)`, puedes estimar κ a partir de la
tasa de fills observada y el spread. Implementa:

```python
def calibrate_kappa(observed_fill_rate, delta):
    """Estima κ tal que exp(-κ·δ) = observed_fill_rate"""
```

Después, verifica: si κ_true=5.0 y δ=SPREAD/2=0.05, la tasa teórica de fill es exp(-5·0.05) ≈ 0.7788.
Calibra κ a partir de esa tasa y recupera 5.0.


In [ ]:
def calibrate_kappa(observed_fill_rate, delta):
    """
    Estima κ dado fill_rate observado y half-spread δ.
    De p = exp(-κ·δ) → κ = -ln(p) / δ
    """
    pass


In [ ]:
# ── Validación E9 ───────────────────────────────────────────────
assert 'calibrate_kappa' in globals()
p_theoretical = np.exp(-5.0 * 0.05)
kappa_est = calibrate_kappa(p_theoretical, 0.05)
assert abs(kappa_est - 5.0) < 1e-9, f"calibrate_kappa recuperó {kappa_est}, esperado 5.0"

# Caso con fill rate = 0.5
kappa_half = calibrate_kappa(0.5, 0.1)
assert abs(kappa_half - np.log(2) / 0.1) < 1e-9
print("E9 ✓ — calibrate_kappa correcta")
print(f"  p=exp(-5·0.05) = {p_theoretical:.4f} → κ estimado = {kappa_est:.4f}")


**Solución E9:**
```python
def calibrate_kappa(observed_fill_rate, delta):
    return -np.log(observed_fill_rate) / delta
```


---
## Ejercicio 10 — Informe completo de performance (Bonus)

Crea una función `performance_report(MMClass, path, T=T_TOTAL, **kwargs)` que retorne
un DataFrame con las siguientes métricas:

| Métrica | Descripción |
|---------|-------------|
| `fills` | Total de fills |
| `final_inv` | Inventario final |
| `pnl` | P&L mark-to-market |
| `inv_std` | Desviación estándar del inventario |
| `max_abs_inv` | Máximo inventario absoluto |
| `fill_rate` | fills / len(path) |
| `pnl_per_fill` | pnl / fills (eficiencia por fill) |

Genera un informe comparando Naive, Skewed y A-S sobre `path_3600`.


In [ ]:
def performance_report(mm_classes, labels, path, T=T_TOTAL, **kwargs):
    """
    Compara múltiples market makers sobre el mismo camino de precios.

    Retorna un DataFrame con una fila por modelo y columnas:
    fills, final_inv, pnl, inv_std, max_abs_inv, fill_rate, pnl_per_fill
    """
    rows = []
    for cls, lbl in zip(mm_classes, labels):
        mm = cls(T=T, seed=42, **kwargs)
        mm.run(path)
        inv_arr = np.array(mm.inv_hist)
        pnl_val = mm.pnl(path[-1])
        rows.append({
            'model': lbl,
            'fills': mm.fills,
            'final_inv': mm.inventory,
            'pnl': round(pnl_val, 4),
            'inv_std': round(float(np.std(inv_arr)), 4),
            'max_abs_inv': int(np.max(np.abs(inv_arr))),
            'fill_rate': round(mm.fills / len(path), 4),
            'pnl_per_fill': round(pnl_val / mm.fills, 4) if mm.fills > 0 else 0
        })
    return pd.DataFrame(rows).set_index('model')

report = performance_report(
    mm_classes=[NaiveMarketMaker, SkewedMarketMaker, ASMarketMaker],
    labels=['Naive', 'Skewed', 'A-S'],
    path=path_3600,
    gamma=0.1
)
print(report.to_string())


In [ ]:
# ── Validación E10 ───────────────────────────────────────────────
assert 'report' in globals() and hasattr(report, 'index')
assert 'fills' in report.columns and 'inv_std' in report.columns
assert 'pnl_per_fill' in report.columns

# A-S debe tener mejor pnl_per_fill que Naive
assert report.loc['A-S', 'pnl_per_fill'] > report.loc['Naive', 'pnl_per_fill'], \
    "A-S debe tener mejor pnl_per_fill que Naive"

# A-S debe tener menor inv_std que Naive
assert report.loc['A-S', 'inv_std'] < report.loc['Naive', 'inv_std'], \
    "A-S debe tener menor inv_std que Naive"

print("E10 ✓ — performance_report correcto")
print(f"  A-S: {report.loc['A-S','pnl_per_fill']:.4f} P&L/fill vs Naive: {report.loc['Naive','pnl_per_fill']:.4f}")


**Solución E10:** El código en las celdas es la solución completa.

---

## Resumen final

| Ejercicio | Concepto | Función/Clase |
|-----------|----------|---------------|
| E1 | Precio de reserva A-S con factor τ | `as_reservation_price` |
| E2 | Half-spread óptimo: dos componentes | `as_optimal_halfspread` |
| E3 | Implementación completa A-S | `ASMarketMaker.run()` |
| E4 | Validación cuantitativa | inv_std < 1, max_abs ≤ 4 |
| E5 | Comparación triple Naive/Skewed/AS | `MarketMakingBacktest.compare()` |
| E6 | Tradeoff γ vs fills/inventario | `df_gamma` |
| E7 | Robustez ante shocks de volatilidad | `price_path_with_shock` |
| E8 | Evolución temporal de δ* | descomposición inv/liq |
| E9 | Calibración de κ desde fills | `calibrate_kappa` |
| E10 | Informe completo de performance | `performance_report` |

> **Para L15 (Exam-Quiz II):** Tendrás que diagnosticar o extender una implementación de MM.
> Asegúrate de entender por qué el orden ask→bid importa para el inventario.
